In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report


import pandas as pd

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ GPU available: {torch.cuda.is_available()}")
print(f"✅ Using device: {device}")
print("\n🎉 You're all set!")

✅ PyTorch version: 2.8.0+cpu
✅ GPU available: False
✅ Using device: cpu

🎉 You're all set!


In [13]:
rootPath = 'C:/Users/deana/Downloads/RetinalProject'
CSVPath = 'C:/Users/deana/Downloads/RetinalProject/data/full_df.csv'
dataPath = 'C:/Users/deana/Downloads/RetinalProject/data/Training Images'

In [14]:
df = pd.read_csv(CSVPath)
print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()

Shape  : (6392, 19)
Columns: ['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename']


,ID,Patient Age,Patient Sex,Left-Fundus,Right-Fundus,Left-Diagnostic Keywords,Right-Diagnostic Keywords,N,D,G,C,A,H,M,O,filepath,labels,target,filename
0,0,69,Female,0_left.jpg,0_right.jpg,cataract,normal fundus,0,0,0,1,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",0_right.jpg
1,1,57,Male,1_left.jpg,1_right.jpg,normal fundus,normal fundus,1,0,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['N'],"[1, 0, 0, 0, 0, 0, 0, 0]",1_right.jpg
2,2,42,Male,2_left.jpg,2_right.jpg,laser spot，moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",2_right.jpg
3,4,53,Male,4_left.jpg,4_right.jpg,macular epiretinal membrane,mild nonproliferative retinopathy,0,1,0,0,0,0,0,1,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",4_right.jpg
4,5,50,Female,5_left.jpg,5_right.jpg,moderate non proliferative retinopathy,moderate non proliferative retinopathy,0,1,0,0,0,0,0,0,../input/ocular-disease-recognition-odir5k/ODI...,['D'],"[0, 1, 0, 0, 0, 0, 0, 0]",5_right.jpg


In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6392 entries, 0 to 6391
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID                         6392 non-null   int64 
 1   Patient Age                6392 non-null   int64 
 2   Patient Sex                6392 non-null   object
 3   Left-Fundus                6392 non-null   object
 4   Right-Fundus               6392 non-null   object
 5   Left-Diagnostic Keywords   6392 non-null   object
 6   Right-Diagnostic Keywords  6392 non-null   object
 7   N                          6392 non-null   int64 
 8   D                          6392 non-null   int64 
 9   G                          6392 non-null   int64 
 10  C                          6392 non-null   int64 
 11  A                          6392 non-null   int64 
 12  H                          6392 non-null   int64 
 13  M                          6392 non-null   int64 
 14  O       

In [17]:
LABEL_COLS = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
LABEL_NAMES = {
    'N': 'Normal', 'D': 'Diabetes', 'G': 'Glaucoma', 'C': 'Cataract',
    'A': 'AMD', 'H': 'Hypertension', 'M': 'Myopia', 'O': 'Other'
}

In [19]:
normal_df  = df[(df['N'] == 1) & (df[LABEL_COLS].sum(axis=1) == 1)].copy()
disease_df = df[df['N'] == 0].copy()

In [20]:
normal_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2101 entries, 1 to 6356
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   ID                         2101 non-null   int64 
 1   Patient Age                2101 non-null   int64 
 2   Patient Sex                2101 non-null   object
 3   Left-Fundus                2101 non-null   object
 4   Right-Fundus               2101 non-null   object
 5   Left-Diagnostic Keywords   2101 non-null   object
 6   Right-Diagnostic Keywords  2101 non-null   object
 7   N                          2101 non-null   int64 
 8   D                          2101 non-null   int64 
 9   G                          2101 non-null   int64 
 10  C                          2101 non-null   int64 
 11  A                          2101 non-null   int64 
 12  H                          2101 non-null   int64 
 13  M                          2101 non-null   int64 
 14  O            